# 05 Model Baselines

Train and compare baseline models directly on the enriched real Zindi
dataset. Save trained artifacts and evaluation metrics for reuse.

In [10]:
%pip install -q pandas numpy scikit-learn lightgbm joblib scipy
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path('..').resolve()))
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import CalibratedClassifierCV

from src.data import load_enriched_raw_competition_tables

ROOT = pathlib.Path('..').resolve()
ZIP_PATH = ROOT / 'data-science-nigeria-challenge-1-loan-default-prediction20250307-26022-im3qg9.zip'
MODELS_DIR = ROOT / 'models'
PROC_DIR = ROOT / 'data' / 'processed'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# 1) Charger les données feature-engineered
df = pd.read_csv(PROC_DIR / 'train_features.csv') if (PROC_DIR / 'train_features.csv').exists() else None
if df is None:
    train_raw, _ = load_enriched_raw_competition_tables(str(ZIP_PATH))
    df = train_raw.copy()

# 2) Hygiène de base
before_rows = len(df)
df = df.drop_duplicates().copy()
after_rows = len(df)

y = df['target_risk_flag'].astype(int)

numeric_all = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != 'target_risk_flag']

# Exclusion explicite des identifiants/variables non causales
id_leakage_candidates = {'customerid', 'systemloanid', 'loannumber', 'loanid', 'application_id', 'customer_id'}
numeric = [c for c in numeric_all if c not in id_leakage_candidates]

# 3) Traitement des manquants et constantes
missing_rate = df[numeric].isna().mean()
kept_missing = [c for c in numeric if missing_rate[c] <= 0.60]

nunique = df[kept_missing].nunique(dropna=True)
kept_variance = [c for c in kept_missing if nunique[c] > 1]

X_num = df[kept_variance].copy()
X_num = X_num.fillna(X_num.median(numeric_only=True)).fillna(0)

# 4) Split AVANT étapes supervisées
Xn_train, Xn_valid, y_train, y_valid = train_test_split(
    X_num, y, test_size=0.2, random_state=42, stratify=y
)

# 5) Outliers: winsorization sur train (1%-99%) puis application valid
winsor_cols = []
for c in Xn_train.columns:
    s = Xn_train[c]
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    if iqr == 0:
        continue
    lo_iqr, hi_iqr = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out_ratio = ((s < lo_iqr) | (s > hi_iqr)).mean()
    if out_ratio > 0.05:
        p1, p99 = np.percentile(s, [1, 99])
        Xn_train[c] = Xn_train[c].clip(p1, p99)
        Xn_valid[c] = Xn_valid[c].clip(p1, p99)
        winsor_cols.append(c)

# 6) Segmentation métier

def build_business_segments(X):
    Z = pd.DataFrame(index=X.index)

    if 'loanamount' in X.columns:
        Z['seg_loanamount'] = pd.cut(
            X['loanamount'],
            bins=[-np.inf, 10000, 30000, np.inf],
            labels=['small', 'medium', 'large']
        ).astype(str)

    if 'termdays' in X.columns:
        Z['seg_termdays'] = pd.cut(
            X['termdays'],
            bins=[-np.inf, 30, 60, np.inf],
            labels=['short', 'mid', 'long']
        ).astype(str)

    if 'due_vs_loan_ratio' in X.columns:
        Z['seg_pricing_ratio'] = pd.cut(
            X['due_vs_loan_ratio'],
            bins=[-np.inf, 1.20, 1.35, np.inf],
            labels=['low_cost', 'standard', 'high_cost']
        ).astype(str)

    if 'customer_age_approx' in X.columns:
        Z['seg_age'] = pd.cut(
            X['customer_age_approx'],
            bins=[-np.inf, 25, 35, 45, np.inf],
            labels=['young', 'early_career', 'mid_career', 'senior']
        ).astype(str)

    if 'max_late_days' in X.columns:
        Z['seg_late_behavior'] = pd.cut(
            X['max_late_days'].fillna(0),
            bins=[-np.inf, 0, 7, 30, np.inf],
            labels=['on_time', 'minor_delay', 'moderate_delay', 'severe_delay']
        ).astype(str)

    return Z

Seg_train = build_business_segments(Xn_train)
Seg_valid = build_business_segments(Xn_valid)

# One-hot des segments métiers
Seg_train_ohe = pd.get_dummies(Seg_train, prefix=Seg_train.columns, dummy_na=False)
Seg_valid_ohe = pd.get_dummies(Seg_valid, prefix=Seg_valid.columns, dummy_na=False)
Seg_valid_ohe = Seg_valid_ohe.reindex(columns=Seg_train_ohe.columns, fill_value=0)

# 7) Dé-corrélation numérique (train uniquement)
corr_matrix = Xn_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [c for c in upper.columns if any(upper[c] > 0.95)]
num_corr_selected = [c for c in Xn_train.columns if c not in to_drop_corr]

X1_train = pd.concat([Xn_train[num_corr_selected], Seg_train_ohe], axis=1)
X1_valid = pd.concat([Xn_valid[num_corr_selected], Seg_valid_ohe], axis=1)

# 8) Screening statistique léger (MI + KS)
mi = mutual_info_classif(X1_train, y_train, random_state=42)
mi_s = pd.Series(mi, index=X1_train.columns)

def ks_score(col):
    a = X1_train.loc[y_train == 0, col]
    b = X1_train.loc[y_train == 1, col]
    if len(a) > 5 and len(b) > 5:
        return ks_2samp(a, b).statistic
    return 0.0

ks_s = pd.Series({c: ks_score(c) for c in X1_train.columns})
stat_keep = [c for c in X1_train.columns if (mi_s[c] >= 0.001) or (ks_s[c] >= 0.03)]
if len(stat_keep) < 8:
    stat_keep = list(X1_train.columns)

X2_train = X1_train[stat_keep]
X2_valid = X1_valid[stat_keep]

# 9) Sélection finale par régularisation L1
l1_selector = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        penalty='l1', solver='liblinear', C=0.20,
        class_weight='balanced', max_iter=5000, random_state=42
    ))
])
l1_selector.fit(X2_train, y_train)
coef = l1_selector.named_steps['clf'].coef_[0]
selected_features = [f for f, w in zip(X2_train.columns, coef) if abs(w) > 1e-8]
if len(selected_features) < 8:
    selected_features = list(X2_train.columns)

X_train = X2_train[selected_features]
X_valid = X2_valid[selected_features]

# Rapport complet
selection_report = pd.DataFrame({
    'stage': [
        'raw_numeric',
        'after_id_filter',
        'after_missing_filter',
        'after_variance_filter',
        'after_corr_filter_numeric',
        'after_add_business_segments',
        'after_stat_screen',
        'after_l1_final',
    ],
    'n_features': [
        len(numeric_all),
        len(numeric),
        len(kept_missing),
        len(kept_variance),
        len(num_corr_selected),
        X1_train.shape[1],
        len(stat_keep),
        len(selected_features),
    ]
})

selection_report.to_csv(MODELS_DIR / 'feature_selection_report.csv', index=False)
pd.Series(selected_features, name='feature').to_csv(MODELS_DIR / 'selected_features.csv', index=False)

print('rows before/after dedup:', before_rows, after_rows)
print('winsorized columns:', winsor_cols)
print(selection_report)
print(f'\nDataset train/valid : {X_train.shape} / {X_valid.shape}')
print(f'Features finales : {len(selected_features)}')

Note: you may need to restart the kernel to use updated packages.
rows before/after dedup: 4368 4368
winsorized columns: ['loanamount', 'totaldue', 'latitude_gps', 'late_payment_rate', 'max_late_days', 'loanamount_safe']
                         stage  n_features
0                  raw_numeric          20
1              after_id_filter          18
2         after_missing_filter          17
3        after_variance_filter          17
4    after_corr_filter_numeric          13
5  after_add_business_segments          29
6            after_stat_screen          26
7               after_l1_final          21

Dataset train/valid : (3494, 21) / (874, 21)
Features finales : 21


In [11]:
def eval_model_with_threshold(name, model, X_tr, y_tr, X_va, y_va):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_va)[:, 1]

    # Seuil optimal selon F1 sur validation
    prec, rec, thr = precision_recall_curve(y_va, proba)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    # prec/rec contiennent un point de plus que thr
    best_idx = int(np.nanargmax(f1[:-1])) if len(thr) > 0 else 0
    best_threshold = float(thr[best_idx]) if len(thr) > 0 else 0.5

    pred_05 = (proba >= 0.5).astype(int)
    pred_best = (proba >= best_threshold).astype(int)

    metrics = {
        'model': name,
        'roc_auc': roc_auc_score(y_va, proba),
        'brier': brier_score_loss(y_va, proba),
        'threshold_best_f1': best_threshold,
        'accuracy@0.5': accuracy_score(y_va, pred_05),
        'precision@0.5': precision_score(y_va, pred_05, zero_division=0),
        'recall@0.5': recall_score(y_va, pred_05, zero_division=0),
        'f1@0.5': f1_score(y_va, pred_05, zero_division=0),
        'accuracy@best': accuracy_score(y_va, pred_best),
        'precision@best': precision_score(y_va, pred_best, zero_division=0),
        'recall@best': recall_score(y_va, pred_best, zero_division=0),
        'f1@best': f1_score(y_va, pred_best, zero_division=0),
        'n_features': X_tr.shape[1],
    }
    return model, metrics

# Modèle final explicable + calibration probabiliste
base_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)),
])

calibrated_model = CalibratedClassifierCV(base_model, method='sigmoid', cv=3)
calibrated_model, m_final = eval_model_with_threshold(
    'logreg_business_calibrated', calibrated_model, X_train, y_train, X_valid, y_valid
)

# CV pour robustesse (AUC)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(base_model, X_train, y_train, scoring='roc_auc', cv=cv, n_jobs=None)
m_final['cv_auc_mean'] = float(np.mean(cv_auc))
m_final['cv_auc_std'] = float(np.std(cv_auc))

results = pd.DataFrame([m_final]).sort_values('roc_auc', ascending=False)
print(results.round(4))

# Importance approximative via coefficients du modèle non calibré (fit pour interprétation)
interpretable_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)),
])
interpretable_model.fit(X_train, y_train)
coef = interpretable_model.named_steps['clf'].coef_[0]
coef_s = pd.Series(coef, index=X_train.columns).sort_values(key=lambda s: s.abs(), ascending=False)
print('\nTop 15 coefficients (abs):')
print(coef_s.head(15))

# Sauvegarde artefacts
joblib.dump({'model': calibrated_model, 'features': selected_features}, MODELS_DIR / 'logreg_raw.pkl')
results.to_csv(MODELS_DIR / 'raw_baselines_metrics.csv', index=False)
coef_s.to_csv(MODELS_DIR / 'model_coefficients.csv', header=['coef'])
(MODELS_DIR / 'best_threshold.txt').write_text(str(m_final['threshold_best_f1']))

print(f"\nSeuil optimal (F1): {m_final['threshold_best_f1']:.4f}")
print(f"Artefacts sauvegardés dans: {MODELS_DIR}")

                        model  roc_auc   brier  threshold_best_f1  \
0  logreg_business_calibrated   0.7277  0.1498             0.2309   

   accuracy@0.5  precision@0.5  recall@0.5  f1@0.5  accuracy@best  \
0        0.7941         0.6316      0.1263  0.2105         0.7529   

   precision@best  recall@best  f1@best  n_features  cv_auc_mean  cv_auc_std  
0           0.443       0.5316   0.4833          21       0.7012      0.0203  

Top 15 coefficients (abs):
due_vs_loan_ratio                  0.553169
max_late_days                      0.549202
late_payment_rate                  0.339032
is_savings_account                 0.287597
ever_late_30d_flag                -0.284902
num_prev_loans                    -0.206562
seg_age_early_career              -0.186043
customer_age_approx               -0.177826
seg_pricing_ratio_low_cost         0.160105
seg_pricing_ratio_standard        -0.160105
loanamount                         0.134838
seg_termdays_mid                   0.062575
ever_lat

In [7]:
import pandas as pd, numpy as np
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import ks_2samp

df = pd.read_csv('../data/processed/train_features.csv')
y = df['target_risk_flag'].astype(int)
num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != 'target_risk_flag']

print('shape:', df.shape)
print('target mean:', round(float(y.mean()), 4))
print('duplicate rows:', int(df.duplicated().sum()))
print('numeric vars:', len(num))

miss = df[num].isna().mean().sort_values(ascending=False)
print('\nmissing >30%:', int((miss > 0.30).sum()))
print(miss.head(10))

nun = df[num].nunique(dropna=True)
print('\nconstant vars:', int((nun <= 1).sum()))

outlier_ratio = {}
for c in num:
    s = df[c].dropna()
    if len(s) < 10:
        outlier_ratio[c] = 0.0
        continue
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    if iqr == 0:
        outlier_ratio[c] = 0.0
        continue
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_ratio[c] = float(((df[c] < lo) | (df[c] > hi)).mean())
out = pd.Series(outlier_ratio).sort_values(ascending=False)
print('\noutlier ratio >5%:', int((out > 0.05).sum()))
print(out.head(10))

corr = df[num].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = []
for col in upper.columns:
    for idx, val in upper[col].dropna().items():
        if val > 0.95:
            pairs.append((idx, col, float(val)))
print('\ncorrelation pairs >0.95:', len(pairs))
print('sample pairs:', pairs[:10])

X = df[num].fillna(df[num].median(numeric_only=True)).fillna(0)
mi = mutual_info_classif(X, y, random_state=42)
mi_s = pd.Series(mi, index=num).sort_values(ascending=False)
print('\nTop MI:')
print(mi_s.head(10))

ks = {}
for c in num:
    a = df.loc[y == 0, c].dropna()
    b = df.loc[y == 1, c].dropna()
    if len(a) > 5 and len(b) > 5:
        ks[c] = float(ks_2samp(a, b).statistic)
    else:
        ks[c] = 0.0
ks_s = pd.Series(ks).sort_values(ascending=False)
print('\nTop KS:')
print(ks_s.head(10))

shape: (4368, 33)
target mean: 0.2179
duplicate rows: 0
numeric vars: 20

missing >30%: 1
days_to_due                        1.00000
ever_late_flag                     0.00206
num_prev_loans                     0.00206
max_late_days                      0.00206
ever_late_30d_flag                 0.00206
late_payment_rate                  0.00206
avg_prev_totaldue_to_loanamount    0.00206
systemloanid                       0.00000
totaldue                           0.00000
loanamount                         0.00000
dtype: float64

constant vars: 1

outlier ratio >5%: 6
latitude_gps           0.220238
max_late_days          0.142857
totaldue               0.096612
loanamount             0.095925
loanamount_safe        0.095925
late_payment_rate      0.088141
num_prev_loans         0.021978
loannumber             0.021978
customer_age_approx    0.014881
longitude_gps          0.011218
dtype: float64

correlation pairs >0.95: 11
sample pairs: [('loanamount', 'totaldue', 0.9943919744421869)